In [1]:
import pandas as pd 
import numpy as np

In [3]:
df = pd.read_csv('Final_Dataset/Dataset_AQI_temp_Clear.csv')
df_copy = df.copy()

In [4]:
df_copy.columns

Index(['datetime', 'PM2.5', 'PM10', 'TSP', 'temp', 'humidity', 'windspeed',
       'winddir', 'sealevelpressure', 'visibility', 'precipitation',
       'solarradiation'],
      dtype='object')

In [5]:
df_copy.rename(columns={'PM2.5':'pm2.5','PM10':'pm10','TSP':'tsp'},inplace=True)

In [7]:
df_copy['datetime']=pd.to_datetime(df_copy['datetime'])

In [8]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2646 entries, 0 to 2645
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   datetime          2646 non-null   datetime64[ns]
 1   pm2.5             2646 non-null   float64       
 2   pm10              2646 non-null   float64       
 3   tsp               2646 non-null   float64       
 4   temp              2646 non-null   float64       
 5   humidity          2646 non-null   float64       
 6   windspeed         2646 non-null   float64       
 7   winddir           2646 non-null   float64       
 8   sealevelpressure  2646 non-null   float64       
 9   visibility        2646 non-null   float64       
 10  precipitation     2646 non-null   float64       
 11  solarradiation    2646 non-null   float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 248.2 KB


In [9]:
df_copy

,datetime,pm2.5,pm10,tsp,temp,humidity,windspeed,winddir,sealevelpressure,visibility,precipitation,solarradiation
0,2016-08-25,24.106910,131.608300,407.084300,77.7,75.7,9.2,226.8,1014.0,4.9,0.0,232.3
1,2016-08-26,20.384295,111.050740,389.271000,74.5,81.3,12.8,171.4,1015.7,4.9,15.5,231.4
2,2016-08-27,16.661680,90.493180,371.457700,77.9,77.4,6.9,162.2,1016.0,5.1,0.0,260.2
3,2016-08-28,31.365190,86.597780,256.126600,75.8,78.8,10.3,223.2,1015.8,5.3,0.0,229.8
4,2016-08-29,35.661620,54.458180,101.626100,74.6,81.8,9.2,289.3,1013.1,4.7,0.5,190.1
...,...,...,...,...,...,...,...,...,...,...,...,...
2641,2024-12-27,90.031286,122.154243,161.637471,52.5,69.4,4.7,160.8,1024.1,3.7,0.0,175.9
2642,2024-12-28,115.462085,150.554242,191.303695,51.5,76.6,4.7,131.1,1023.8,2.8,2.0,67.2
2643,2024-12-29,104.691319,139.022639,201.824028,54.8,68.3,11.4,257.4,1021.4,3.3,1.6,137.5
2644,2024-12-30,81.418264,107.658472,139.620417,52.4,71.1,8.1,204.3,1020.5,4.1,0.0,150.8


In [12]:

# Official US EPA AQI Breakpoints


# PM2.5 AQI breakpoints
pm25_breakpoints = [
    (0.0, 12.0, 0, 50),
    (12.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 150.4, 151, 200),
    (150.5, 250.4, 201, 300),
    (250.5, 350.4, 301, 400),
    (350.5, 500.4, 401, 500)
]

# PM10 AQI breakpoints
pm10_breakpoints = [
    (0, 54, 0, 50),
    (55, 154, 51, 100),
    (155, 254, 101, 150),
    (255, 354, 151, 200),
    (355, 424, 201, 300),
    (425, 504, 301, 400),
    (505, 604, 401, 500)
]



# Function to calculate pollutant sub-index AQI


def calculate_subindex(concentration, breakpoints):

    # Handle missing values safely
    if pd.isna(concentration):
        return np.nan

    # Find appropriate breakpoint range
    for bp_lo, bp_hi, i_lo, i_hi in breakpoints:

        if bp_lo <= concentration <= bp_hi:

            # AQI Formula
            aqi = (
                (i_hi - i_lo) / (bp_hi - bp_lo)
            ) * (
                concentration - bp_lo
            ) + i_lo

            return round(aqi)

    # Return NaN if concentration exceeds defined range
    return np.nan



# Function to calculate final AQI for each row


def calculate_aqi(row):

    # PM2.5 AQI
    pm25_aqi = calculate_subindex(
        row['pm2.5'],
        pm25_breakpoints
    )

    # PM10 AQI
    pm10_aqi = calculate_subindex(
        row['pm10'],
        pm10_breakpoints
    )

    # Final AQI is maximum sub-index
    return max(pm25_aqi, pm10_aqi)



# Create AQI Column


df_copy['aqi'] = df_copy.apply(calculate_aqi, axis=1)



# Optional: AQI Category Labels


def categorize_aqi(aqi):

    if aqi <= 50:
        return 'Good'

    elif aqi <= 100:
        return 'Moderate'

    elif aqi <= 150:
        return 'Unhealthy for Sensitive Groups'

    elif aqi <= 200:
        return 'Unhealthy'

    elif aqi <= 300:
        return 'Very Unhealthy'

    else:
        return 'Hazardous'


df_copy['aqi_category'] = df_copy['aqi'].apply(categorize_aqi)




print(df_copy[['pm2.5', 'pm10', 'aqi', 'aqi_category']].head())

       pm2.5       pm10    aqi                    aqi_category
0  24.106910  131.60830   89.0                        Moderate
1  20.384295  111.05074   79.0                        Moderate
2  16.661680   90.49318   69.0                        Moderate
3  31.365190   86.59778   92.0                        Moderate
4  35.661620   54.45818  101.0  Unhealthy for Sensitive Groups


In [21]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2646 entries, 0 to 2645
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   datetime          2646 non-null   datetime64[ns]
 1   pm2.5             2646 non-null   float64       
 2   pm10              2646 non-null   float64       
 3   tsp               2646 non-null   float64       
 4   temp              2646 non-null   float64       
 5   humidity          2646 non-null   float64       
 6   windspeed         2646 non-null   float64       
 7   winddir           2646 non-null   float64       
 8   sealevelpressure  2646 non-null   float64       
 9   visibility        2646 non-null   float64       
 10  precipitation     2646 non-null   float64       
 11  solarradiation    2646 non-null   float64       
 12  aqi               2639 non-null   float64       
 13  aqi_category      2646 non-null   object        
dtypes: datetime64[ns](1), fl

In [20]:
df_copy[df_copy['aqi'].isnull()].shape[0]

7

In [22]:
df_copy[df_copy['aqi'].isnull()]

,datetime,pm2.5,pm10,tsp,temp,humidity,windspeed,winddir,sealevelpressure,visibility,precipitation,solarradiation,aqi,aqi_category
191,2017-03-04,35.413815,38.513780,44.963895,60.0,48.9,17.9,275.3,1013.8,4.4,0.0,245.5,NaN,Hazardous
492,2017-12-30,35.499510,38.125350,38.804860,51.6,67.0,9.2,158.9,1021.0,3.0,0.0,173.2,NaN,Hazardous
1362,2020-06-08,12.023860,16.613720,35.734330,73.2,82.5,10.3,267.6,1010.8,5.4,0.0,259.8,NaN,Hazardous
1576,2021-04-23,35.404920,95.600000,267.600000,67.7,47.0,12.8,291.9,1016.1,5.7,0.0,329.3,NaN,Hazardous
2114,2022-12-27,55.425072,62.685577,90.658334,51.0,63.9,11.4,267.4,1017.2,4.3,0.0,171.8,NaN,Hazardous
2321,2023-07-22,12.070833,13.379167,14.542014,75.9,83.3,13.9,176.7,1016.3,4.4,3.0,203.2,NaN,Hazardous
2338,2023-08-08,12.008333,13.320833,14.094982,73.4,90.9,8.1,170.3,1007.3,4.1,64.0,34.2,NaN,Hazardous


In [23]:
df_copy.to_csv('Final_dataset.csv',index=False)